In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import keras
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import keras.models as km
from keras.models import Sequential
from keras import activations, initializers, regularizers, constraints
from keras.layers import Dense, Activation


In [4]:
df = sns.load_dataset("penguins")
df


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


In [5]:
df.shape

(344, 7)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    object 
 1   island             344 non-null    object 
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    object 
dtypes: float64(4), object(3)
memory usage: 18.9+ KB


In [7]:
df.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


In [8]:
df.tail()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,NaN
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female
343,Gentoo,Biscoe,49.9,16.1,213.0,5400.0,Male


In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
df.isnull().sum()

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64

In [11]:

df['sex'].value_counts()


sex
Male      168
Female    165
Name: count, dtype: int64

In [12]:

df['species'].value_counts()

species
Adelie       152
Gentoo       124
Chinstrap     68
Name: count, dtype: int64

In [13]:

df['island'].value_counts()

island
Biscoe       168
Dream        124
Torgersen     52
Name: count, dtype: int64

In [15]:

df['bill_length_mm'].value_counts()


bill_length_mm
41.1    7
45.2    6
39.6    5
50.5    5
50.0    5
       ..
35.6    1
36.8    1
43.1    1
38.5    1
49.9    1
Name: count, Length: 164, dtype: int64

In [16]:
df['bill_depth_mm'].value_counts()

bill_depth_mm
17.0    12
18.6    10
17.9    10
15.0    10
18.5    10
        ..
13.2     1
14.9     1
21.5     1
20.2     1
17.4     1
Name: count, Length: 80, dtype: int64

In [17]:
df["sex"].fillna("male",inplace=True)

C:\Users\Asus\AppData\Local\Temp\ipykernel_18180\232772562.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["sex"].fillna("male",inplace=True)


In [18]:
df['bill_length_mm'] = df['bill_length_mm'].fillna(41.1)

In [19]:
df['bill_depth_mm'] = df['bill_depth_mm'].fillna(17.0)

In [20]:
df['island'] = df['island'].fillna('Biscoe')

In [21]:
df['species'] = df['species'].fillna('Adelie')

In [22]:
df.isnull().sum()

species              0
island               0
bill_length_mm       0
bill_depth_mm        0
flipper_length_mm    2
body_mass_g          2
sex                  0
dtype: int64

In [23]:

df['flipper_length_mm'] = df['flipper_length_mm'].fillna(df['flipper_length_mm'].mode()[0])
df['body_mass_g'] = df['body_mass_g'].fillna(df['body_mass_g'].mode()[0])



In [24]:
df.isnull().sum()

species              0
island               0
bill_length_mm       0
bill_depth_mm        0
flipper_length_mm    0
body_mass_g          0
sex                  0
dtype: int64

In [37]:
# 1. Create a copy and label encode the categories directly using pandas
df_encoded = df.copy()
df_encoded['species'] = df_encoded['species'].astype('category').cat.codes
df_encoded['island'] = df_encoded['island'].astype('category').cat.codes
df_encoded['sex'] = df_encoded['sex'].astype('category').cat.codes

# 2. Define your X and y from the encoded dataframe
y = df_encoded['body_mass_g']
X = df_encoded.drop(columns=['body_mass_g'])

# 3. Do the Train-Test Split using the new numeric X
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.3, random_state=123)

# 4. Scale the features (This will now run perfectly!)
sc = StandardScaler()
train_X = sc.fit_transform(train_X)
test_X = sc.transform(test_X)


In [38]:
train_X.shape

(240, 6)

In [46]:
model = Sequential()
model.add(Dense(units = 12, kernel_initializer = 'uniform', activation = 'relu', input_dim = 6))
model.add(Dense(units = 6, kernel_initializer = 'uniform', activation = 'relu'))
model.add(Dense(units = 4, kernel_initializer = 'uniform', activation = 'relu'))
model.add(Dense(units = 1, kernel_initializer = 'uniform', activation = 'linear'))

C:\Users\Asus\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [40]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 11)             │            77 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │            28 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │             5 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 182 (728.00 B)

 Trainable params: 182 (728.00 B)

 Non-trainable params: 0 (0.00 B)

In [47]:
model.compile(loss='mean_squared_error', optimizer='adam',metrics=["mae"])

In [42]:
print(train_X.shape)
print(train_y.shape)

(240, 6)
(240,)


In [48]:
model.fit(train_X, train_y, epochs=150, batch_size=10, validation_split=0.1)

# Evaluate the model
mse = model.evaluate(test_X, test_y)
print(f'Test Mean Squared Error: {mse}')

Epoch 1/150
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - loss: 18440200.0000 - mae: 4217.2339 - val_loss: 18642630.0000 - val_mae: 4223.9341
Epoch 2/150
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 18439966.0000 - mae: 4217.2070 - val_loss: 18642362.0000 - val_mae: 4223.9023
Epoch 3/150
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 18439630.0000 - mae: 4217.1685 - val_loss: 18641926.0000 - val_mae: 4223.8530
Epoch 4/150
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 18439024.0000 - mae: 4217.0991 - val_loss: 18641046.0000 - val_mae: 4223.7563
Epoch 5/150
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 18437764.0000 - mae: 4216.9614 - val_loss: 18639230.0000 - val_mae: 4223.5610
Epoch 6/150
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 18435196.0000 - mae: 4216.6831 - val_loss: 18635640.0000 - val_mae: 4223.1792
Epoch 7/150
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 18430474.0000 - mae: 4216.1738 - val_loss: 18629146.0000 - val_mae: 4222.4946
Epoch 8/150
22/22 ━━━━━━━━━━━━━━━━━━━━ 0

In [49]:
predictions = model.predict(test_X)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step


In [53]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# 1. Get predictions and flatten them
predictions = model.predict(test_X).flatten()
actual_answers = test_y.values

# 2. Calculate the metrics
mse = mean_squared_error(actual_answers, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(actual_answers, predictions)

# 3. Print the results clearly
print("--- MODEL PERFORMANCE METRICS ---")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} grams")
print(f"R-squared Score (R2): {r2:.2f}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
--- MODEL PERFORMANCE METRICS ---
Mean Squared Error (MSE): 320367.44
Root Mean Squared Error (RMSE): 566.01 grams
R-squared Score (R2): 0.43
